# 27. Integrated network automation capstones

**Level:** Advanced

**Learning outcome:** Combine domain models, policies, collection, change planning, and honest per-target reporting.

**Prerequisites:** Complete the foundational and relevant workflow notebooks before attempting these capstones.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](26_Topology_Graphs_and_Address_Planning.ipynb) | [Next notebook](28_Equality_Hashing_and_Operator_Overloading.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Concept notes and mental model

**Level:** Advanced

**Learning goal:** Combine the models into reviewable offline workflows with explicit success and failure outcomes.

**Capstone architecture.** Keep domain models, transport boundaries, policies, orchestration,
and reporting distinct. The following worked patterns combine multiple earlier concepts
rather than introducing a giant framework. All data and outcomes are synthetic. No command
is sent to a real device, no package installation is needed, and no credentials are used.

**A reviewable change flow.** Validate intent -> observe a baseline -> compute a plan -> obtain
approval for that exact plan -> apply through a controlled adapter -> collect fresh state ->
verify postconditions -> record an accurate outcome. If application or verification fails,
attempt the documented recovery and report whether recovery itself succeeded.

**Fleet outcomes are per-device.** One failed read must not become an empty successful report.
Keep target identity with the result, distinguish expected transport errors from programming
errors, and bound concurrency. Preserve enough diagnostics for the operator without copying
sensitive payloads into logs.

**Telemetry pipelines have units and freshness.** A calculation is only meaningful if samples
belong to the same interface identity, counters are comparable, and timestamps are trustworthy.
A compliance engine should distinguish violated policy from missing or malformed evidence.

**Beyond this notebook.** A real project needs supported library versions, reviewed platform
adapters, authorization, secrets management, reliable persistence, CI, observability, and
integration tests in an approved lab. These notes teach object design and reasoning. They do
not replace vendor documentation or an organization's change process.

For each capstone, sketch responsibilities first, run all ten datasets, identify the invariants,
then implement the extension prompt. Prefer a small explicit design you can explain to another
engineer over clever metaprogramming.

**Pause and explain:** Walk one capstone from input through result and identify the exact trust and side-effect boundaries.

## Worked-example structure

This notebook contains **5 worked patterns and 50 executable scenario examples**. Each pattern explains its purpose, logic, common mistakes, and extension exercise. Its ten scenarios vary the input data; they are related practice, not ten unrelated algorithms. Original Example IDs are retained so you can locate the same exercise in `AI_oops_learning.ipynb`.

The input table lists only fields needed by that pattern. Expected output is shown before each code cell and a saved executed result appears afterward. Expected rejection branches are checked explicitly; unexpected exceptions are not silently ignored.

## Pattern 096: Capstone: reviewed idempotent MTU deployment

**Core concept:** Combine an immutable intent, a plan, approval, reconciliation, and postcheck.

**Network-engineering purpose:** An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

### Read the implementation in this order
1. Represent desired MTU with a frozen dataclass.
2. Compute whether baseline differs.
3. Require approval before any mutation.
4. Verify the postcondition and check a repeated approved apply is unchanged.

**Common mistake / boundary:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

**Practice after the ten worked scenarios:** Bind approval to baseline plus intent and reject drift between plan and apply.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0951 - Capstone: reviewed idempotent MTU deployment: branch scenario

**Scenario 01/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['unchanged', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [1]:
# Example 0951: Capstone: reviewed idempotent MTU deployment - branch
case = {'mtu': 1500, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['unchanged', 'unchanged']


### Example 0952 - Capstone: reviewed idempotent MTU deployment: campus scenario

**Scenario 02/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [2]:
# Example 0952: Capstone: reviewed idempotent MTU deployment - campus
case = {'mtu': 9000, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0953 - Capstone: reviewed idempotent MTU deployment: dc scenario

**Scenario 03/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 9216 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [3]:
# Example 0953: Capstone: reviewed idempotent MTU deployment - dc
case = {'mtu': 9216, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 9216 audit: ['changed', 'unchanged']


### Example 0954 - Capstone: reviewed idempotent MTU deployment: wan scenario

**Scenario 04/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 576 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [4]:
# Example 0954: Capstone: reviewed idempotent MTU deployment - wan
case = {'mtu': 576, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 576 audit: ['changed', 'unchanged']


### Example 0955 - Capstone: reviewed idempotent MTU deployment: lab scenario

**Scenario 05/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [5]:
# Example 0955: Capstone: reviewed idempotent MTU deployment - lab
case = {'mtu': 1600, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0956 - Capstone: reviewed idempotent MTU deployment: wireless scenario

**Scenario 06/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['unchanged', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [6]:
# Example 0956: Capstone: reviewed idempotent MTU deployment - wireless
case = {'mtu': 1500, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['unchanged', 'unchanged']


### Example 0957 - Capstone: reviewed idempotent MTU deployment: remote scenario

**Scenario 07/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [7]:
# Example 0957: Capstone: reviewed idempotent MTU deployment - remote
case = {'mtu': 2000, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0958 - Capstone: reviewed idempotent MTU deployment: core scenario

**Scenario 08/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 4470 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [8]:
# Example 0958: Capstone: reviewed idempotent MTU deployment - core
case = {'mtu': 4470, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 4470 audit: ['changed', 'unchanged']


### Example 0959 - Capstone: reviewed idempotent MTU deployment: mgmt scenario

**Scenario 09/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `approved` | `False` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 1500 audit: ['denied']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [9]:
# Example 0959: Capstone: reviewed idempotent MTU deployment - mgmt
case = {'mtu': 9100, 'approved': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 1500 audit: ['denied']


### Example 0960 - Capstone: reviewed idempotent MTU deployment: backup scenario

**Scenario 10/10; pattern 096; Advanced.** This is a self-contained offline exercise. An engineer can trace exactly when the simulated state changes and why a second apply is a no-op.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `approved` | `True` |

**Logic to trace:** Represent desired MTU with a frozen dataclass. Verify the postcondition and check a repeated approved apply is unchanged. Follow the class-specific steps under Pattern 096 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
observed: 8192 audit: ['changed', 'unchanged']
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This local state transition has no real-device atomicity, authorization, or durable audit guarantee.

In [10]:
# Example 0960: Capstone: reviewed idempotent MTU deployment - backup
case = {'mtu': 8192, 'approved': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Intent:
    mtu: int

    def __post_init__(self):
        if type(self.mtu) is not int or not 576 <= self.mtu <= 9216:
            raise ValueError("invalid MTU")

class Deployment:
    def __init__(self):
        self.observed, self.audit = 1500, []

    def plan(self, intent):
        return None if self.observed == intent.mtu else (self.observed, intent.mtu)

    def apply(self, intent, *, approved):
        if not approved:
            self.audit.append("denied")
            raise PermissionError("approval required")
        plan = self.plan(intent)
        if plan is not None:
            self.observed = intent.mtu
        if self.observed != intent.mtu:
            raise RuntimeError("postcheck failed")
        self.audit.append("changed" if plan else "unchanged")
        return plan is not None

deployment, intent = Deployment(), Intent(case["mtu"])
before = deployment.observed
try:
    deployment.apply(intent, approved=case["approved"])
except PermissionError:
    assert not case["approved"] and deployment.observed == before
else:
    assert deployment.observed == intent.mtu
    assert deployment.apply(intent, approved=True) is False
print("observed:", deployment.observed, "audit:", deployment.audit)


observed: 8192 audit: ['changed', 'unchanged']


## Pattern 097: Capstone: fleet collection with structured outcomes

**Core concept:** A result object preserves target identity and explicit transport failures.

**Network-engineering purpose:** Fleet reporting should not hide one failed read behind an aggregate success count.

### Read the implementation in this order
1. Use a frozen outcome record.
2. Inject a replay transport.
3. Catch only the expected TimeoutError per target.
4. Assert the successful and failed result shapes separately.

**Common mistake / boundary:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

**Practice after the ten worked scenarios:** Add bounded concurrency while preserving a stable input-to-output mapping.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0961 - Capstone: fleet collection with structured outcomes: branch scenario

**Scenario 01/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='branch-edge-01', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [11]:
# Example 0961: Capstone: fleet collection with structured outcomes - branch
case = {'hostname': 'branch-edge-01'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='branch-edge-01', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0962 - Capstone: fleet collection with structured outcomes: campus scenario

**Scenario 02/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='campus-access-02', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [12]:
# Example 0962: Capstone: fleet collection with structured outcomes - campus
case = {'hostname': 'campus-access-02'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='campus-access-02', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0963 - Capstone: fleet collection with structured outcomes: dc scenario

**Scenario 03/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='dc-leaf-03', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [13]:
# Example 0963: Capstone: fleet collection with structured outcomes - dc
case = {'hostname': 'dc-leaf-03'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='dc-leaf-03', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0964 - Capstone: fleet collection with structured outcomes: wan scenario

**Scenario 04/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='wan-border-04', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [14]:
# Example 0964: Capstone: fleet collection with structured outcomes - wan
case = {'hostname': 'wan-border-04'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='wan-border-04', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0965 - Capstone: fleet collection with structured outcomes: lab scenario

**Scenario 05/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='lab-router-05', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [15]:
# Example 0965: Capstone: fleet collection with structured outcomes - lab
case = {'hostname': 'lab-router-05'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='lab-router-05', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0966 - Capstone: fleet collection with structured outcomes: wireless scenario

**Scenario 06/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='wireless-dist-06', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [16]:
# Example 0966: Capstone: fleet collection with structured outcomes - wireless
case = {'hostname': 'wireless-dist-06'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='wireless-dist-06', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0967 - Capstone: fleet collection with structured outcomes: remote scenario

**Scenario 07/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='remote-gateway-07', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [17]:
# Example 0967: Capstone: fleet collection with structured outcomes - remote
case = {'hostname': 'remote-gateway-07'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='remote-gateway-07', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0968 - Capstone: fleet collection with structured outcomes: core scenario

**Scenario 08/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='core-router-08', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [18]:
# Example 0968: Capstone: fleet collection with structured outcomes - core
case = {'hostname': 'core-router-08'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='core-router-08', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0969 - Capstone: fleet collection with structured outcomes: mgmt scenario

**Scenario 09/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='mgmt-switch-09', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [19]:
# Example 0969: Capstone: fleet collection with structured outcomes - mgmt
case = {'hostname': 'mgmt-switch-09'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='mgmt-switch-09', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


### Example 0970 - Capstone: fleet collection with structured outcomes: backup scenario

**Scenario 10/10; pattern 097; Advanced.** This is a self-contained offline exercise. Fleet reporting should not hide one failed read behind an aggregate success count.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Use a frozen outcome record. Assert the successful and failed result shapes separately. Follow the class-specific steps under Pattern 097 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Outcome(hostname='backup-edge-10', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Unexpected exceptions should still fail loudly rather than be mislabeled as ordinary device failure.

In [20]:
# Example 0970: Capstone: fleet collection with structured outcomes - backup
case = {'hostname': 'backup-edge-10'}

from dataclasses import dataclass

@dataclass(frozen=True)
class Outcome:
    hostname: str
    state: str | None
    error: str | None

class Replay:
    def read(self, hostname):
        if hostname == "lab-unavailable":
            raise TimeoutError("read timeout")
        return "up"

class FleetCollector:
    def __init__(self, transport):
        self.transport = transport

    def collect(self, names):
        results = []
        for name in names:
            try:
                state = self.transport.read(name)
            except TimeoutError as error:
                results.append(Outcome(name, None, str(error)))
            else:
                results.append(Outcome(name, state, None))
        return results

results = FleetCollector(Replay()).collect([case["hostname"], "lab-unavailable"])
assert results[0] == Outcome(case["hostname"], "up", None)
assert results[1] == Outcome("lab-unavailable", None, "read timeout")
assert sum(item.error is None for item in results) == 1
print(results)


[Outcome(hostname='backup-edge-10', state='up', error=None), Outcome(hostname='lab-unavailable', state=None, error='read timeout')]


## Pattern 098: Capstone: rate calculation and health policy

**Core concept:** Separate immutable samples, unit-aware calculation, and threshold evaluation.

**Network-engineering purpose:** A telemetry report can show both the measured bps value and the policy result.

### Read the implementation in this order
1. Represent each timestamped counter sample.
2. Reject non-increasing time and counter resets.
3. Calculate bits per second.
4. Evaluate against an injected policy limit and assert both data and classification.

**Common mistake / boundary:** These synthetic samples assume no wrap, no reset, and the same interface identity.

**Practice after the ten worked scenarios:** Add a result state for unavailable evidence without converting it to a healthy zero rate.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0971 - Capstone: rate calculation and health policy: branch scenario

**Scenario 01/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `1000` |
| `traffic_bytes` | `0` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 0.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [21]:
# Example 0971: Capstone: rate calculation and health policy - branch
case = {'counter': 1000, 'traffic_bytes': 0}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 0.0 health: ok


### Example 0972 - Capstone: rate calculation and health policy: campus scenario

**Scenario 02/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `2000` |
| `traffic_bytes` | `500` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 2000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [22]:
# Example 0972: Capstone: rate calculation and health policy - campus
case = {'counter': 2000, 'traffic_bytes': 500}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 2000.0 health: ok


### Example 0973 - Capstone: rate calculation and health policy: dc scenario

**Scenario 03/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `3000` |
| `traffic_bytes` | `1000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 4000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [23]:
# Example 0973: Capstone: rate calculation and health policy - dc
case = {'counter': 3000, 'traffic_bytes': 1000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 4000.0 health: ok


### Example 0974 - Capstone: rate calculation and health policy: wan scenario

**Scenario 04/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `4000` |
| `traffic_bytes` | `1001` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 4004.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [24]:
# Example 0974: Capstone: rate calculation and health policy - wan
case = {'counter': 4000, 'traffic_bytes': 1001}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps:

 4004.0 health: high


### Example 0975 - Capstone: rate calculation and health policy: lab scenario

**Scenario 05/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `5000` |
| `traffic_bytes` | `2000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 8000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [25]:
# Example 0975: Capstone: rate calculation and health policy - lab
case = {'counter': 5000, 'traffic_bytes': 2000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps:

 8000.0 health: high


### Example 0976 - Capstone: rate calculation and health policy: wireless scenario

**Scenario 06/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `6000` |
| `traffic_bytes` | `250` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 1000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [26]:
# Example 0976: Capstone: rate calculation and health policy - wireless
case = {'counter': 6000, 'traffic_bytes': 250}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps:

 1000.0 health: ok


### Example 0977 - Capstone: rate calculation and health policy: remote scenario

**Scenario 07/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `7000` |
| `traffic_bytes` | `3000` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 12000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [27]:
# Example 0977: Capstone: rate calculation and health policy - remote
case = {'counter': 7000, 'traffic_bytes': 3000}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps:

 12000.0 health: high


### Example 0978 - Capstone: rate calculation and health policy: core scenario

**Scenario 08/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `8000` |
| `traffic_bytes` | `750` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 3000.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [28]:
# Example 0978: Capstone: rate calculation and health policy - core
case = {'counter': 8000, 'traffic_bytes': 750}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 3000.0 health: ok


### Example 0979 - Capstone: rate calculation and health policy: mgmt scenario

**Scenario 09/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `9000` |
| `traffic_bytes` | `1500` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 6000.0 health: high
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [29]:
# Example 0979: Capstone: rate calculation and health policy - mgmt
case = {'counter': 9000, 'traffic_bytes': 1500}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 6000.0 health: high


### Example 0980 - Capstone: rate calculation and health policy: backup scenario

**Scenario 10/10; pattern 098; Advanced.** This is a self-contained offline exercise. A telemetry report can show both the measured bps value and the policy result.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `counter` | `10000` |
| `traffic_bytes` | `100` |

**Logic to trace:** Represent each timestamped counter sample. Evaluate against an injected policy limit and assert both data and classification. Follow the class-specific steps under Pattern 098 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
bps: 400.0 health: ok
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** These synthetic samples assume no wrap, no reset, and the same interface identity.

In [30]:
# Example 0980: Capstone: rate calculation and health policy - backup
case = {'counter': 10000, 'traffic_bytes': 100}

from dataclasses import dataclass

@dataclass(frozen=True)
class Sample:
    seconds: int
    byte_count: int

class Rate:
    def calculate(self, old, new):
        elapsed, delta = new.seconds - old.seconds, new.byte_count - old.byte_count
        if elapsed <= 0 or delta < 0:
            raise ValueError("incomparable samples")
        return delta * 8 / elapsed

class HealthPolicy:
    def __init__(self, limit):
        self.limit = limit

    def evaluate(self, bps):
        return "ok" if bps <= self.limit else "high"

old = Sample(0, case["counter"])
new = Sample(2, case["counter"] + case["traffic_bytes"])
bps = Rate().calculate(old, new)
result = HealthPolicy(4000).evaluate(bps)
assert bps == case["traffic_bytes"] * 4
assert result == ("ok" if case["traffic_bytes"] <= 1000 else "high")
print("bps:", bps, "health:", result)


bps: 400.0 health: ok


## Pattern 099: Capstone: composable compliance evidence

**Core concept:** Individual checks return explicit evidence and an engine aggregates it.

**Network-engineering purpose:** An interface can fail one requirement without losing the evidence for checks that passed.

### Read the implementation in this order
1. Define a frozen check result.
2. Implement separate MTU and admin-state checks.
3. Inject checks into an engine.
4. Verify each result and the overall all-pass condition.

**Common mistake / boundary:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

**Practice after the ten worked scenarios:** Extend the result with PASS/FAIL/UNKNOWN and define aggregation semantics.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0981 - Capstone: composable compliance evidence: branch scenario

**Scenario 01/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [31]:
# Example 0981: Capstone: composable compliance evidence - branch
case = {'mtu': 1500, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]


### Example 0982 - Capstone: composable compliance evidence: campus scenario

**Scenario 02/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9000` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9000), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [32]:
# Example 0982: Capstone: composable compliance evidence - campus
case = {'mtu': 9000, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9000), Evidence(check='admin', passed=False, observed=False)]


### Example 0983 - Capstone: composable compliance evidence: dc scenario

**Scenario 03/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9216` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9216), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [33]:
# Example 0983: Capstone: composable compliance evidence - dc
case = {'mtu': 9216, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9216), Evidence(check='admin', passed=True, observed=True)]


### Example 0984 - Capstone: composable compliance evidence: wan scenario

**Scenario 04/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `576` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=False, observed=576), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [34]:
# Example 0984: Capstone: composable compliance evidence - wan
case = {'mtu': 576, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=False, observed=576), Evidence(check='admin', passed=True, observed=True)]


### Example 0985 - Capstone: composable compliance evidence: lab scenario

**Scenario 05/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1600` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1600), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [35]:
# Example 0985: Capstone: composable compliance evidence - lab
case = {'mtu': 1600, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1600), Evidence(check='admin', passed=False, observed=False)]


### Example 0986 - Capstone: composable compliance evidence: wireless scenario

**Scenario 06/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `1500` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [36]:
# Example 0986: Capstone: composable compliance evidence - wireless
case = {'mtu': 1500, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=1500), Evidence(check='admin', passed=True, observed=True)]


### Example 0987 - Capstone: composable compliance evidence: remote scenario

**Scenario 07/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `2000` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=2000), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [37]:
# Example 0987: Capstone: composable compliance evidence - remote
case = {'mtu': 2000, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=2000), Evidence(check='admin', passed=False, observed=False)]


### Example 0988 - Capstone: composable compliance evidence: core scenario

**Scenario 08/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `4470` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=4470), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [38]:
# Example 0988: Capstone: composable compliance evidence - core
case = {'mtu': 4470, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=4470), Evidence(check='admin', passed=True, observed=True)]


### Example 0989 - Capstone: composable compliance evidence: mgmt scenario

**Scenario 09/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `9100` |
| `healthy` | `False` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=9100), Evidence(check='admin', passed=False, observed=False)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [39]:
# Example 0989: Capstone: composable compliance evidence - mgmt
case = {'mtu': 9100, 'healthy': False}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=9100), Evidence(check='admin', passed=False, observed=False)]


### Example 0990 - Capstone: composable compliance evidence: backup scenario

**Scenario 10/10; pattern 099; Advanced.** This is a self-contained offline exercise. An interface can fail one requirement without losing the evidence for checks that passed.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `mtu` | `8192` |
| `healthy` | `True` |

**Logic to trace:** Define a frozen check result. Verify each result and the overall all-pass condition. Follow the class-specific steps under Pattern 099 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[Evidence(check='mtu', passed=True, observed=8192), Evidence(check='admin', passed=True, observed=True)]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** Missing evidence needs an explicit unknown/error state in production, not an automatic pass.

In [40]:
# Example 0990: Capstone: composable compliance evidence - backup
case = {'mtu': 8192, 'healthy': True}

from dataclasses import dataclass

@dataclass(frozen=True)
class Evidence:
    check: str
    passed: bool
    observed: object

class MtuCheck:
    def run(self, record):
        return Evidence("mtu", record["mtu"] >= 1500, record["mtu"])

class AdminCheck:
    def run(self, record):
        return Evidence("admin", record["admin_up"] is True, record["admin_up"])

class Engine:
    def __init__(self, checks):
        self.checks = tuple(checks)

    def evaluate(self, record):
        return [check.run(record) for check in self.checks]

record = {"mtu": case["mtu"], "admin_up": case["healthy"]}
results = Engine([MtuCheck(), AdminCheck()]).evaluate(record)
assert results[0].passed == (case["mtu"] >= 1500)
assert results[1].passed == case["healthy"]
assert all(item.passed for item in results) == (case["mtu"] >= 1500 and case["healthy"])
print(results)


[Evidence(check='mtu', passed=True, observed=8192), Evidence(check='admin', passed=True, observed=True)]

## Pattern 100: Capstone: validated inventory import and report

**Core concept:** A schema boundary, immutable record, duplicate check, and report form one small pipeline.

**Network-engineering purpose:** A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

### Read the implementation in this order
1. Validate field names and nonempty strings.
2. Create a frozen record through from_mapping.
3. Reject a duplicate before repository mutation.
4. Export a deterministic public report.

**Common mistake / boundary:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

**Practice after the ten worked scenarios:** Add address validation and a versioned JSON import/export format with round-trip tests.

The following ten examples intentionally reuse this implementation with explicit local scenario data. The displayed output comes from running each full code cell; assertions also check behavior, including expected rejections where present.

### Example 0991 - Capstone: validated inventory import and report: branch scenario

**Scenario 01/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'branch'` |
| `hostname` | `'branch-edge-01'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "branch-edge-01", "site": "branch"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [41]:
# Example 0991: Capstone: validated inventory import and report - branch
case = {'site': 'branch', 'hostname': 'branch-edge-01'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "branch-edge-01", "site": "branch"}]


### Example 0992 - Capstone: validated inventory import and report: campus scenario

**Scenario 02/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'campus'` |
| `hostname` | `'campus-access-02'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "campus-access-02", "site": "campus"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [42]:
# Example 0992: Capstone: validated inventory import and report - campus
case = {'site': 'campus', 'hostname': 'campus-access-02'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "campus-access-02", "site": "campus"}]

### Example 0993 - Capstone: validated inventory import and report: dc scenario

**Scenario 03/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'dc'` |
| `hostname` | `'dc-leaf-03'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "dc-leaf-03", "site": "dc"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [43]:
# Example 0993: Capstone: validated inventory import and report - dc
case = {'site': 'dc', 'hostname': 'dc-leaf-03'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "dc-leaf-03", "site": "dc"}]

### Example 0994 - Capstone: validated inventory import and report: wan scenario

**Scenario 04/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wan'` |
| `hostname` | `'wan-border-04'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "wan-border-04", "site": "wan"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [44]:
# Example 0994: Capstone: validated inventory import and report - wan
case = {'site': 'wan', 'hostname': 'wan-border-04'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "wan-border-04", "site": "wan"}]

### Example 0995 - Capstone: validated inventory import and report: lab scenario

**Scenario 05/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'lab'` |
| `hostname` | `'lab-router-05'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "lab-router-05", "site": "lab"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [45]:
# Example 0995: Capstone: validated inventory import and report - lab
case = {'site': 'lab', 'hostname': 'lab-router-05'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "lab-router-05", "site": "lab"}]


### Example 0996 - Capstone: validated inventory import and report: wireless scenario

**Scenario 06/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'wireless'` |
| `hostname` | `'wireless-dist-06'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "wireless-dist-06", "site": "wireless"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [46]:
# Example 0996: Capstone: validated inventory import and report - wireless
case = {'site': 'wireless', 'hostname': 'wireless-dist-06'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "wireless-dist-06", "site": "wireless"}]


### Example 0997 - Capstone: validated inventory import and report: remote scenario

**Scenario 07/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'remote'` |
| `hostname` | `'remote-gateway-07'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "remote-gateway-07", "site": "remote"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [47]:
# Example 0997: Capstone: validated inventory import and report - remote
case = {'site': 'remote', 'hostname': 'remote-gateway-07'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "remote-gateway-07", "site": "remote"}]


### Example 0998 - Capstone: validated inventory import and report: core scenario

**Scenario 08/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'core'` |
| `hostname` | `'core-router-08'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "core-router-08", "site": "core"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [48]:
# Example 0998: Capstone: validated inventory import and report - core
case = {'site': 'core', 'hostname': 'core-router-08'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "core-router-08", "site": "core"}]


### Example 0999 - Capstone: validated inventory import and report: mgmt scenario

**Scenario 09/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'mgmt'` |
| `hostname` | `'mgmt-switch-09'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "mgmt-switch-09", "site": "mgmt"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [49]:
# Example 0999: Capstone: validated inventory import and report - mgmt
case = {'site': 'mgmt', 'hostname': 'mgmt-switch-09'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "mgmt-switch-09", "site": "mgmt"}]


### Example 1000 - Capstone: validated inventory import and report: backup scenario

**Scenario 10/10; pattern 100; Advanced.** This is a self-contained offline exercise. A network inventory import should reject malformed or duplicated records instead of silently overwriting them.

**Local inputs** (all are defined again in the code cell):

| Input | Value |
| --- | --- |
| `site` | `'backup'` |
| `hostname` | `'backup-edge-10'` |

**Logic to trace:** Validate field names and nonempty strings. Export a deterministic public report. Follow the class-specific steps under Pattern 100 for the intermediate operations. Predict the result from these inputs before running the cell.

**Expected printed output:**
```text
[{"hostname": "backup-edge-10", "site": "backup"}]
```

**What verifies it:** the assertions in the next cell check returned values, state, interactions, or explicit expected exceptions, not just whether the code prints. An assertion failure indicates an unexpected result. If you intentionally change input or behavior, update your prediction and relevant tests rather than deleting the checks.

**Remember:** This repository is in memory and its validation is intentionally narrower than a complete inventory schema.

In [50]:
# Example 1000: Capstone: validated inventory import and report - backup
case = {'site': 'backup', 'hostname': 'backup-edge-10'}

from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Device:
    hostname: str
    site: str

    @classmethod
    def from_mapping(cls, record):
        if set(record) != {"hostname", "site"}:
            raise ValueError("invalid fields")
        if not all(isinstance(value, str) and value.strip() for value in record.values()):
            raise ValueError("nonempty strings required")
        return cls(record["hostname"].strip().lower(), record["site"].strip().lower())

class Inventory:
    def __init__(self):
        self._devices = {}

    def add(self, device):
        if device.hostname in self._devices:
            raise ValueError("duplicate hostname")
        self._devices[device.hostname] = device

    def report(self):
        return [{"hostname": item.hostname, "site": item.site}
                for item in sorted(self._devices.values(), key=lambda item: item.hostname)]

device = Device.from_mapping({"hostname": case["hostname"], "site": case["site"]})
inventory = Inventory()
inventory.add(device)
try:
    inventory.add(device)
except ValueError as error:
    assert str(error) == "duplicate hostname"
else:
    raise AssertionError("duplicate overwritten")
assert inventory.report() == [{"hostname": case["hostname"], "site": case["site"]}]
print(json.dumps(inventory.report(), sort_keys=True))


[{"hostname": "backup-edge-10", "site": "backup"}]


## Check your understanding: Integrated network automation capstones

**You should now be able to:** Combine domain models, policies, collection, change planning, and honest per-target reporting.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/tutorial/classes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.